---
## 7.5. BPR Hyperparameter Tuning

BPR was evaluated earlier with default hyperparameters and lost to SVD
(21.6% vs 50.8% Precision@1). Before finalizing model selection, we give
BPR a fair tuning budget to confirm whether the gap closes.


In [ ]:
# ── Tune BPR hyperparameters with Optuna ───────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_USERS_BPR = active_users[:100]

def bpr_objective(trial):
    kf    = trial.suggest_int('k', 16, 128)
    lr    = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    reg   = trial.suggest_float('lambda_reg', 0.0001, 0.05, log=True)
    iters = trial.suggest_int('max_iter', 50, 200)

    m = BPR(k=kf, max_iter=iters, learning_rate=lr,
             lambda_reg=reg, seed=42, verbose=False)
    m.fit(cornac_data)

    hits = []
    for uid in TUNE_USERS_BPR:
        seen = train_by_user.get(uid, set())
        pos_list = list(eval_test.get(uid, set()))
        if not pos_list: continue
        pos = pos_list[0]
        neg_pool = list(known_rids - seen - eval_test.get(uid, set()))
        if len(neg_pool) < 99: continue
        cands = [pos] + random.sample(neg_pool, 99)
        try:
            preds = sorted(
                [(r, m.score(cornac_data.uid_map[uid], cornac_data.iid_map[r]))
                 for r in cands if r in cornac_data.iid_map],
                key=lambda x: x[1], reverse=True
            )
            rank = [p[0] for p in preds].index(pos) + 1
            hits.append(int(rank <= 10))
        except:
            continue
    return np.mean(hits) if hits else 0.0

print('Tuning BPR (30 trials)...')
bpr_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
bpr_study.optimize(bpr_objective, n_trials=30)

bpr_best = bpr_study.best_params
print(f'\nBest BPR Hit@10 (tuning): {bpr_study.best_value:.1%}')
for k, v in bpr_best.items(): print(f'  {k:<15}={v}')


In [ ]:
# ── Retrain BPR with tuned hyperparameters ─────────────────────────────────────
print('Retraining BPR with tuned hyperparameters...')
bpr_tuned = BPR(k=bpr_best['k'], max_iter=bpr_best['max_iter'],
                  learning_rate=bpr_best['learning_rate'],
                  lambda_reg=bpr_best['lambda_reg'], seed=42, verbose=True)
bpr_tuned.fit(cornac_data)

def bpr_tuned_score(uid_str, rid_str):
    try:
        return bpr_tuned.score(cornac_data.uid_map[uid_str], cornac_data.iid_map[rid_str])
    except KeyError:
        return -999.0

print('BPR (tuned) trained.')

with open('models/bpr_tuned.pkl','wb') as f:
    pickle.dump({'model':bpr_tuned,'params':bpr_best},f)


In [ ]:
# ── Re-evaluate tuned BPR with the same protocol as before ────────────────────
print('=== BPR (tuned) — Method 1 ===')
bpr_tuned_prec, bpr_tuned_rec = eval_method1(bpr_tuned_score, 'BPR (tuned)')


In [ ]:
# ── Final three-way comparison: SVD vs BPR (default) vs BPR (tuned) ───────────
print('=== Final Comparison: SVD vs BPR (default) vs BPR (tuned) ===')
print(f'{"k":<5}{"SVD":>10}{"BPR-default":>14}{"BPR-tuned":>12}{"Winner":>14}')
for k in K_VALUES:
    s  = np.mean(svd_prec[k])
    bd = np.mean(bpr_prec[k])
    bt = np.mean(bpr_tuned_prec[k])
    winner = max([('SVD',s), ('BPR-default',bd), ('BPR-tuned',bt)], key=lambda x: x[1])[0]
    print(f'{k:<5}{s:>10.4f}{bd:>14.4f}{bt:>12.4f}{winner:>14}')

all_candidates = {
    'svd':         np.mean(svd_prec[1]),
    'bpr_default': np.mean(bpr_prec[1]),
    'bpr_tuned':   np.mean(bpr_tuned_prec[1]),
}
winner_name = max(all_candidates, key=all_candidates.get)
print(f'\n🏆 Overall winner: {winner_name}  ({all_candidates[winner_name]:.1%})')

ACTIVE_CF_MODEL = 'svd' if winner_name == 'svd' else 'bpr'
if winner_name == 'bpr_tuned':
    bpr = bpr_tuned
    bpr_score = bpr_tuned_score
    best = bpr_best  # used downstream by Section 8/12 summary

validate(all_candidates[winner_name] > 0.10,
         'Winning model beats random baseline (k=1)',
         '> 10% (random baseline)',
         f'{all_candidates[winner_name]:.1%}')


In [ ]:
# ── Plot: SVD vs BPR (default) vs BPR (tuned) ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(K_VALUES, [np.mean(svd_prec[k]) for k in K_VALUES],
         marker='o', color=C_PURPLE, linewidth=2, markersize=7, label='SVD')
ax.plot(K_VALUES, [np.mean(bpr_prec[k]) for k in K_VALUES],
         marker='^', color=C_FLAG, linewidth=2, markersize=7, label='BPR (default)')
ax.plot(K_VALUES, [np.mean(bpr_tuned_prec[k]) for k in K_VALUES],
         marker='D', color=C_AFTER, linewidth=2, markersize=7, label='BPR (tuned)')
ax.plot(K_VALUES, [k/100 for k in K_VALUES],
         marker='s', color='gray', linestyle='--', markersize=5, label='Random')
ax.set_xlabel('k'); ax.set_ylabel('Precision@k')
ax.set_title('Model Comparison: SVD vs BPR (default) vs BPR (tuned)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots/svd_vs_bpr_tuned.png', dpi=120, bbox_inches='tight')
plt.show()
